In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [9]:
!pip install rdkit -q

from rdkit import Chem
from rdkit.Chem import Draw

smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"
mol = Chem.MolFromSmiles(smiles)

if mol is not None:
    print("Success! Molecule successfully created.")
    display(mol)
else:
    print("Invalid SMILES string!")

Success! Molecule successfully created.


In [10]:
from rdkit import Chem
from rdkit.Chem import Descriptors

smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"
mol = Chem.MolFromSmiles(smiles)

if mol is not None:
    
    mol_wt = Descriptors.MolWt(mol)
    log_p = Descriptors.MolLogP(mol)
    h_donors = Descriptors.NumHDonors(mol)
    h_acceptors = Descriptors.NumHAcceptors(mol)

    print("--- Molecular Properties Report ---")
    print(f"Molecule SMILES: {smiles}")
    print(f"Molecular Weight: {mol_wt:.2f} g/mol")
    print(f"LogP (Lipophilicity): {log_p:.2f}")
    print(f"Hydrogen Bond Donors: {h_donors}")
    print(f"Hydrogen Bond Acceptors: {h_acceptors}")
else:
    print("Invalid SMILES string!")

--- Molecular Properties Report ---
Molecule SMILES: CC(=O)OC1=CC=CC=C1C(=O)O
Molecular Weight: 180.16 g/mol
LogP (Lipophilicity): 1.31
Hydrogen Bond Donors: 1
Hydrogen Bond Acceptors: 3


In [11]:
from rdkit import Chem
from rdkit.Chem import Descriptors

molecules_data = [
    {"Name": "Aspirin", "SMILES": "CC(=O)OC1=CC=CC=C1C(=O)O"},
    {"Name": "Caffeine", "SMILES": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"},
    {"Name": "Paracetamol", "SMILES": "CC(=O)NC1=CC=C(O)C=C1"}
]

print("--- Drug-Likeness Screening Report (Lipinski's Rule of Five) ---\n")

for item in molecules_data:
    mol = Chem.MolFromSmiles(item["SMILES"])

    if mol is not None:
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        h_donors = Descriptors.NumHDonors(mol)
        h_acceptors = Descriptors.NumHAcceptors(mol)

        if (mw <= 500) and (logp <= 5) and (h_donors <= 5) and (h_acceptors <= 10):
            status = "PASSED (Good Drug-Like Properties)"
        else:
            status = "FAILED"

        print(f"Molecule: {item['Name']}")
        print(f" -MolWt: {mw:.2f}, LogP: {logp:.2f}, Donors: {h_donors}, Acceptors: {h_acceptors}")
        print(f" - Status: {status}\n")

--- Drug-Likeness Screening Report (Lipinski's Rule of Five) ---

Molecule: Aspirin
 -MolWt: 180.16, LogP: 1.31, Donors: 1, Acceptors: 3
 - Status: PASSED (Good Drug-Like Properties)

Molecule: Caffeine
 -MolWt: 194.19, LogP: -1.03, Donors: 0, Acceptors: 3
 - Status: PASSED (Good Drug-Like Properties)

Molecule: Paracetamol
 -MolWt: 151.16, LogP: 1.35, Donors: 2, Acceptors: 2
 - Status: PASSED (Good Drug-Like Properties)



In [12]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

molecules_data = [
    {"Name": "Aspirin", "SMILES": "CC(=O)OC1=CC=CC=C1C(=O)O"},
    {"Name": "Caffeine", "SMILES": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"},
    {"Name": "Paracetamol", "SMILES": "CC(=O)NC1=CC=C(O)C=C1"}
]

results_list = []

for item in molecules_data:
    mol = Chem.MolFromSmiles(item["SMILES"])
    if mol is not None:
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        h_donors = Descriptors.NumHDonors(mol)
        h_acceptors = Descriptors.NumHAcceptors(mol)

        status = "PASSED" if (mw <= 500) and (logp <=5) and (h_donors <= 5) and (h_acceptors <= 10) else "FAILED"

        results_list.append({
            "Molecule Name": item["Name"],
            "SMILES": item["SMILES"],
            "Mol Weight": round(mw, 2),
            "LogP": round(logp, 2),
            "H_Donors": h_donors,
            "H_Acceptors": h_acceptors,
            "Lipinski Status": status
        })

df_report = pd.DataFrame(results_list)

print("--- FINAL CHEMINFORMATICS REPORT TABLE ---")
display(df_report)

df_report.to_csv("chem_insights_report.csv", index=False)
print("\nSuccess! Report saved as 'chem_insights_report.csv' in your workspace.")

--- FINAL CHEMINFORMATICS REPORT TABLE ---


,Molecule Name,SMILES,Mol Weight,LogP,H_Donors,H_Acceptors,Lipinski Status
0,Aspirin,CC(=O)OC1=CC=CC=C1C(=O)O,180.16,1.31,1,3,PASSED
1,Caffeine,CN1C=NC2=C1C(=O)N(C(=O)N2C)C,194.19,-1.03,0,3,PASSED
2,Paracetamol,CC(=O)NC1=CC=C(O)C=C1,151.16,1.35,2,2,PASSED



Success! Report saved as 'chem_insights_report.csv' in your workspace.


In [13]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.DataStructs import TanimotoSimilarity

smiles_1 = "CC(=O)OC1=CC=CC=C1C(=O)O"
smiles_2 = "CC(=O)NC1=CC=C(O)C=C1"

mol1 = Chem.MolFromSmiles(smiles_1)
mol2 = Chem.MolFromSmiles(smiles_2)

fp1 = AllChem.GetMorganFingerprintAsBitVect(mol1, 2, nBits=1024)
fp2 = AllChem.GetMorganFingerprintAsBitVect(mol2, 2, nBits=1024)

similarity_score = TanimotoSimilarity(fp1, fp2)

print("--- Molecular Similarity Analysis ---")
print(f"Molecule 1: Aspirin")
print(f"Molecule 2: Paracetamol")
print(f"Tanimoto Similarity Score: {similarity_score:.2f}")

if similarity_score > 0.5:
    print("Result: High structural similarity!")
else:
    print("Result: Low to moderate structural similarity (different core scaffolds).")

--- Molecular Similarity Analysis ---
Molecule 1: Aspirin
Molecule 2: Paracetamol
Tanimoto Similarity Score: 0.23
Result: Low to moderate structural similarity (different core scaffolds).


[21:29:21] DEPRECATION WARNING: please use MorganGenerator
[21:29:21] DEPRECATION WARNING: please use MorganGenerator


In [14]:

!pip install rdkit -q

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

def batch_analyze_from_csv(file_path):
    """
    Reads a CSV file containing molecule names and SMILES strings,
    computes physicochemical properties, and screens for Lipinski compliance.
    """
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found. Please upload your CSV.")
        return None
    
    results = []
    for index, row in df.iterrows():
        name = row.get("Compound Name", f"Compound_{index}")
        smi = row.get("SMILES")
        
        mol = Chem.MolFromSmiles(str(smi))
        if mol is not None:
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            h_donors = Descriptors.NumHDonors(mol)
            h_acceptors = Descriptors.NumHAcceptors(mol)
            
            # Lipinski Rule Check
            status = "PASSED" if (mw <= 500) and (logp <= 5) and (h_donors <= 5) and (h_acceptors <= 10) else "FAILED"
            
            results.append({
                "Compound Name": name,
                "SMILES": smi,
                "Mol Weight": round(mw, 2),
                "LogP": round(logp, 2),
                "H-Donors": h_donors,
                "H-Acceptors": h_acceptors,
                "Lipinski Status": status
            })
            
    return pd.DataFrame(results)

# Pehle wali save ki gayi file ko load karein
analyzed_df = batch_analyze_from_csv("chem_insights_report.csv")
if analyzed_df is not None:
    display(analyzed_df)

,Compound Name,SMILES,Mol Weight,LogP,H-Donors,H-Acceptors,Lipinski Status
0,Compound_0,CC(=O)OC1=CC=CC=C1C(=O)O,180.16,1.31,1,3,PASSED
1,Compound_1,CN1C=NC2=C1C(=O)N(C(=O)N2C)C,194.19,-1.03,0,3,PASSED
2,Compound_2,CC(=O)NC1=CC=C(O)C=C1,151.16,1.35,2,2,PASSED


# 🧪 ChemInsights: Automated Molecular Property Explorer & Drug-Likeness Screener

## 📌 Project Overview
**ChemInsights** is a Python-based cheminformatics tool designed to automate the early-stage screening of chemical structures. It helps researchers, students, and medicinal chemists quickly calculate key physicochemical properties, check Lipinski's Rule of Five compliance, and analyze structural similarities between molecules using RDKit and Pandas.

## 🚀 Key Features
1. **SMILES Parsing & Validation:** Safely parses chemical structures from SMILES strings.
2. **Physicochemical Property Calculator:** Computes Molecular Weight, LogP (Lipophilicity), H-Bond Donors, and H-Bond Acceptors.
3. **Lipinski's Rule of Five Screening:** Automatically filters molecules based on oral drug-likeness standards.
4. **Structural Similarity Search:** Calculates Tanimoto Similarity using Morgan Fingerprints to compare compound scaffolds.
5. **Report Generation:** Exports automated screening results into a clean CSV report (`chem_insights_report.csv`).

## 🛠️ Tech Stack & Libraries Used
* **Python** (Core programming logic)
* **RDKit** (Cheminformatics & molecular descriptor calculations)
* **Pandas** (Data manipulation and table generation)
* **Matplotlib** (Structure visualization)

## 📊 Sample Output
| Molecule Name | SMILES | Mol Weight | LogP | H_Donors | H_Acceptors | Lipinski Status |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Aspirin** | `CC(=O)OC1=CC=CC=C1C(=O)O` | 180.16 | 1.31 | 1 | 3 | PASSED |
| **Caffeine** | `CN1C=NC2=C1C(=O)N(C(=O)N2C)C` | 194.19 | -1.03 | 0 | 3 | PASSED |
| **Paracetamol** | `CC(=O)NC1=CC=C(O)C=C1` | 151.16 | 1.35 | 2 | 2 | PASSED |